# Spectral Data Cubes from Galaxy Particle Distributions

In this example we show the main high-level workflow for creating a spectral data cube from particle data.

We first load some demo CAMELS data and a grid, generate per-particle spectra, construct an ``IntegratedFieldUnit``, and then call the galaxy cube getter. A lower-level ``SpectralCube`` example is shown afterwards for custom workflows.

In [ ]:
import time

import numpy as np
from astropy.cosmology import Planck18 as cosmo
from unyt import angstrom, kpc

from synthesizer import TEST_DATA_DIR
from synthesizer.emission_models import IntrinsicEmission
from synthesizer.grid import Grid
from synthesizer.imaging import SpectralCube
from synthesizer.instruments import IntegratedFieldUnit
from synthesizer.kernel_functions import Kernel
from synthesizer.load_data.load_camels import load_CAMELS_IllustrisTNG

# Define the grid
grid_name = "test_grid"
grid = Grid(grid_name)

# Create galaxy object
gal = load_CAMELS_IllustrisTNG(
    TEST_DATA_DIR,
    snap_name="camels_snap.hdf5",
    group_name="camels_subhalo.hdf5",
    physical=True,
)[0]

# Calculate the stellar rest frame SEDs for all particles in erg / s / Hz
model = IntrinsicEmission(grid, fesc=0.1, per_particle=True)
sed = gal.stars.get_spectra(model)

# Calculate the observed SED in nJy
sed.get_fnu(cosmo, gal.redshift)

print(gal.stars)

## Spectral Data Cube Creation

We now have most of the ingredients we need to generate a spectral data cube from our galaxy. The remaining ingredients are the wavelength array, the spatial resolution, the field of view, and a kernel for smoothing particles over. We will define those below and then construct an ``IntegratedFieldUnit`` to drive the high-level cube generation path.

In [ ]:
# Define the width of the image
width = 30 * kpc

# Define image resolution (here we arbitrarily set it to 100
# pixels along an axis)
resolution = width / 200

# Define the wavelength array
lam = np.linspace(10**3.5, 10**4.5, 1000)

print(
    "Data cube spatial width is %.2f kpc with a %.2f kpc spaxel resolution"
    % (width.value, resolution.value)
)

# Get the SPH kernel
kernel = Kernel()
kernel_data = kernel.get_kernel()

Synthesizer allows you to make either a histogram data cube, where particles are sorted into individual spaxels, or a smoothed data cube, where particles are distributed over their SPH kernels. We will begin with the recommended high-level path on the galaxy, where an ``IntegratedFieldUnit`` defines the wavelength sampling and spatial resolution.

The possible cube quantities are `"lnu"`, `"luminosity"` or `"llam"` for rest-frame luminosities, or `"fnu"`, `"flam"` or `"flux"` for fluxes. Here we will make a cube populated with `"flux"`.

In [ ]:
cube_start = time.time()

# Construct the IFU that defines the cube geometry
ifu = IntegratedFieldUnit(
    label="DemoIFU",
    resolution=resolution,
    lam=lam * angstrom,
)

# Generate the cube via the high-level galaxy helper
cube = gal.get_data_cube(
    fov=width,
    instrument=ifu,
    cube_type="smoothed",
    stellar_spectra="intrinsic",
    kernel=kernel_data,
    kernel_threshold=1,
    quantity="flux",
)

print("Spectral data cube created, took:", time.time() - cube_start)

And that's it. We now have a spectral data cube to analyse. We can visualise the data cube by making an animation. 

In [ ]:
# Animate the data cube
ani = cube.animate_data_cube(fps=240, show=True)

### Direct ``SpectralCube`` interface

If you need more direct control, you can also work with the lower-level ``SpectralCube`` object. In that path you construct the cube yourself and call its ``generate_data_cube_*`` methods explicitly.

In [ ]:
cube = SpectralCube(resolution=resolution, lam=lam * angstrom, fov=width)
cube.generate_data_cube_smoothed(
    sed,
    coordinates=gal.stars.centered_coordinates,
    smoothing_lengths=gal.stars.smoothing_lengths,
    kernel=kernel_data,
    kernel_threshold=1,
    quantity="fnu",
)